In [ ]:
import subprocess
import warnings
warnings.filterwarnings('ignore')
import re
import os
from pathlib import Path
import yaml
import pandas as pd
import duckdb
import shutil
import json
import os
import snowflake.connector
from datetime import datetime

### **BUILD/MIGRATE NEW PLATFORM DBT OBJECTS**

1. **Segregate Gold, Silver, Bronze models.**
    - We first load manifest.json and from it find out details of all sources, refs etc. Then we segregate the models.
        1. _BRONZE_ 
            -  Holds All models that fetch data from source aka Landing zone with new names.
            -  New table alias format: [ods schema name]_TABLE_NAME. Example: new alias: SAPAPI_ZCDV_T_CMTU024_ACDOCA, CRM_FILE_OPPORTUNITY.
            -  dbt Model name format: [database-name]__[schema_name]\__[table_alias].
        2. _SILVER_
            -   Holds all models that fetch data from bronze only.
            -   dbt Model name format: [database-name]__[schema_name]\__[table_alias].
            -   In case of duplicate model name scenario, schema is added to the model name. (Example: Company table is present in mdm and myskills, make it Mdm_company and Myskills_company).
        3. _GOLD_
            -   Holds all models that fetch data from bronze or source.
            -   dbt Model name format: [database-name]__[schema_name]\__[table_alias].
            -   In case of duplicate model name scenario, schema is added to the model name. (Example: Company table is present in mdm and myskills, make it Mdm_company and Myskills_company).

2. **Generate ALL Objects Lineage for the DEs to be migrated.**
    - For Service we picked 4 DEs and generated four data frames : __final_df_models__, __final_df_sources__, __final_df_model_names__, and, __final_df_common_vs_pub__, these four df holds information of all the models, sources, new vs old model arch models, and common vs pub models required to build a DE(basically the whole lineage master data).
    -   There are some sources like EDW_DE, PUB_COMMON_SYSTEMS etc (refer dict: *schema_mappings*) which are not refreshed and are static in snowflake. To simplify migration and better consolidation of these, we created a new source schema: *STATIC_OBJECTS* which will hold all objects which are one time loads and doest refresh(This also includes all OTL Legacy base tables).
    -   We replace the source references with new *STATIC_OBJETCS* schema in the new dbt models generated.

3.  **New dbt Models Generation.**
    - We create 4 dictionaries all_models_dict, bronze_models_dict, silver_models_dict, gold_models_dict from final_df_model_names
    -   Here we run through the dictionaries and pick the old model sql , replace the old refs, alias, model_name  with the new ones using the mapping and finally write the SQL file into the new dbt project location.
    -   Refs that use de_operations are converted to source. (Example: PUB_DE_OPERATIONS_D_EMPLOYEE is converted to source('DE_OPEARATIONS', 'D_EMPLOYEE').)

4.  **New dbt Tests Generation.**
    - There are currently two generic tests that should be applied universally to all models:

        - recently_run: Checks dbt_artifacts to see if the dbt model was executed in a given interval. This defaults to the last 24 hours.
        - has_data: Checks the row count of the Snowflake table and raises a failure if there is no data.
    
    - There is also one test that we expect for all BRONZE tables:

        - expect_row_values_to_have_recent_data: This is an imported dbt_expectations  test that is attached to the _FILEDATE column. If _FILEDATE does not exist it will raise an error. Similar to the recently_run test it looks an interval. One difference, however is that this interval defaults to 1 day. This does not give the level of accuracy we are looking for though so we need to pass in parameters for 24 hours.

5.  **Build new Objects in New Databases (Clone Prod).**
    -   Using new_names mapping dict we generate clone scripts for our new objects and create those in snowflake, basically we clone from prod.
6.  **Create OLD V/s NEW MAPPING Document.**
    -   As we created new objects with new names in new locations, it would be indispensable to provide users with a mapping document that outlines the old and new name association. Using *new_names* dictionary and a *result_dict* variable which we created when we segregated gold models - we generate a mapping excel. (PLEASE ADD LEGACY OTLS VIEWS IN THIS DOCUMENT MANUALLY)
7.  **Parrallel Testing.**
    -   using the MAPPING Document, we hit Snowflake to fetch the counts for old and new objects and generate a report (adding 3 new column in existing mapping doc- old_count,new_count,COUNT_DIFF). 
8.  **Snow file Generation.**
    -   Use the code to generate DDLs to create or clone tables.

----------------------------------------------WIKI ENDS------------------------------------------

Segregate Gold, Silver, Bronze models Starts

In [ ]:
de_paths = ['DE_EHS']

project_dir = "C:\\Sources\\Repo\\EnterpriseCloud\\astro_datautility\\dags\\dbt\\dbt_project" 
pub_path = os.path.join(project_dir,"models\pub")

new_schema_name = "TEH_EHS"

In [ ]:
#1.GET LIST OF REPORTS TO BE MIGRATED INTO NEW DBT PROJECT
pub_de = []
for de in de_paths:
    de_path = os.path.join(pub_path,de)
    for file in os.listdir(de_path):
        if "PUB__" in file:
            file = file.replace(".SQL",'')
            file = file.replace(".sql",'')
            pub_de.append(file)
            print(file)

In [ ]:
#2.LOAD DBT OBJECT
from dbt.cli.main import dbtRunner,dbtRunnerResult
from dbt.contracts.graph.manifest import Manifest

res: dbtRunnerResult = dbtRunner().invoke(["parse","--project-dir",project_dir])
manifest: Manifest = res.result

dbt = dbtRunner(manifest=manifest)

In [ ]:
#3. LOAD MANIFEST AS THIS WILL BE USED TO GENERATE NEW NAME MAPPING AND THEN THE NEW MODELS AS WELL
with open(os.path.join(project_dir,"target\\manifest.json"), 'r') as f:
    manifest_json = json.load(f)

In [ ]:
node_dict = {}

for node in manifest_json['nodes'].values():
    node_dict[node['name']] = node

In [ ]:
schema_mappings = {
    "EDW__DE": "STATIC_OBJECTS",
    "PUB__COMMON_SYSTEM": "STATIC_OBJECTS",
    "COMMON__DE_SAP_MASTER_DATA": "STATIC_OBJECTS",
    "ODS__KRONOS": "STATIC_OBJECTS",
    "ODS__MDM": "STATIC_OBJECTS",
    "COMMON__TEACOMMON": "STATIC_OBJECTS",
    "ADMIN__SECURITY": "STATIC_OBJECTS",
    "ODS__BAAN": "STATIC_OBJECTS"
}

schema_mappings_list = ["EDW__DE",
    "PUB__COMMON_SYSTEM",
    "COMMON__DE_SAP_MASTER_DATA",
    "ODS__KRONOS",
    "ODS__MDM",
    "COMMON__TEACOMMON",
    "ADMIN__SECURITY",
    "ODS__BAAN"]

In [ ]:
from collections import deque

final_df_models = pd.DataFrame()
final_df_sources = pd.DataFrame()
final_df_model_names = pd.DataFrame()
final_df_common_vs_pub = pd.DataFrame()

all_models_queue = deque(pub_de)
all_models_set = set(pub_de)
all_sources = set()
all_common_vs_pub = set()

de_models = []
de_sources = []
de_model_names = []
de_common_vs_pub = []

discarded_models = []

while(len(all_models_queue) > 0):
    model_name = all_models_queue.popleft()
    if model_name in node_dict:
        node = node_dict[model_name]
    else:
        print('Model name not present in node dict ' + model_name)

    bronze_check = 0
    silver_check = 0
    gold_check = 0
    
    database = node['database'].replace('_DEV', '').upper()
    schema = node['schema'].upper()
    table = node['alias'].upper()

    for ref in node['refs']:
        ref_model_name  = ref['name']
        if ref_model_name not in all_models_set:
            all_models_queue.append(ref_model_name)
            all_models_set.add(ref_model_name)
        if 'common__' in ref_model_name.lower():
            if 'pub__' in model_name.lower() and len(node['refs']) == 1:
                gold_check = 0
                table
                if f"{ref_model_name},{model_name}" not in all_common_vs_pub:
                    all_common_vs_pub.add(f"{ref_model_name}.{model_name}")
                    if [ref_model_name, model_name] not in de_common_vs_pub:
                        de_common_vs_pub.append([ref_model_name, model_name])

            else:
                gold_check = 1
        if 'ods__' in ref_model_name.lower():
            silver_check = 1

        if 'PUB__DE_OPERATIONS' in ref_model_name:
            source_schema = 'DE_OPERATIONS'
            source_table = ref_model_name.split('__')[-1]

            if f"{source_schema}.{source_table}" not in all_sources:
                all_sources.add(f"{source_schema}.{source_table}")
                de_sources.append([source_schema, source_table])

    for source in node['sources']:
        source_schema = source[0]
        source_table = source[1]
        if source_schema not in schema_mappings_list:
            bronze_check = 1
        else:
            source_schema = schema_mappings[source_schema]

        if ".".join(source) not in all_sources:
            all_sources.add(".".join(source))
            de_sources.append([source_schema, source_table])


    if bronze_check == 1:
        new_database = 'BRONZE'
        new_table = f"{schema}_{table}"
        new_model_name = '__'.join([new_database,new_schema_name, new_table])

        de_models.append([database, schema, model_name])
        de_model_names.append([new_database, model_name, new_model_name])
    elif gold_check == 1:
        new_database = 'GOLD'
        new_model_name = '__'.join([new_database,new_schema_name, table])

        de_models.append([database, schema, model_name])
        de_model_names.append([new_database, model_name, new_model_name])
    elif silver_check == 1:
        new_database = 'SILVER'
        new_model_name = '__'.join([new_database,new_schema_name, table])
        
        de_models.append([database, schema, model_name])
        de_model_names.append([new_database, model_name, new_model_name])
    else:
        discarded_models.append(model_name)

print(f'discarded_models - {len(discarded_models)}')

In [ ]:
# Removing de_operations models since it will be used as source in new arch.
de_common_vs_pub = [row for row in de_common_vs_pub if 'PUB__DE_OPERATIONS' not in row[1]]

In [ ]:
# Handle Model names where pub view alias is different that common model alias.

diff_model_names = [row for row in de_common_vs_pub if row[-1].split('__')[-1] != row[-2].split('__')[-1]]
for diff_model_name in diff_model_names:
    for index in range(len(de_model_names)):
        if de_model_names[index][1] == diff_model_name[0]:
            new_model_alias = diff_model_name[1].split('__')[-1]

            old_model_name = de_model_names[index][2]
            new_model_name_list = old_model_name.split('__')
            new_model_name_list[-1] = new_model_alias
            new_model_name = "__".join(new_model_name_list)

            de_model_names[index][2] = new_model_name   
            print(f'{old_model_name} changed to {new_model_name}.')

In [ ]:
# Handle Duplicate Model Names

model_occurences = {}

for database, old_model, new_model in de_model_names:
    if new_model not in model_occurences:
        model_occurences[new_model] = 0

    model_occurences[new_model] += 1

duplicate_models = []

for model_name, occurences in model_occurences.items():
    if occurences > 1:
        duplicate_models.append(model_name)
        print(model_name, occurences)



for i in range(len(de_model_names)):
    database = de_model_names[i][0]
    old_model = de_model_names[i][1]
    new_model = de_model_names[i][2]
    if new_model in duplicate_models:
        new_model_prefix = "__".join(new_model.split('__')[:2])
        new_table = "_".join(old_model.split('__')[1:])
        new_model_name = new_model_prefix + '__' + new_table
        print(f"{new_model}|{old_model}|{new_model_name}")
        de_model_names[i][2] = new_model_name


Segregate Gold, Silver, Bronze models Ends

Generate ALL Objects Lineage for the DEs to be migrated Starts

In [ ]:
df_models = pd.DataFrame(de_models,columns=['database','schema','model'])
df_sources = pd.DataFrame(de_sources,columns=['schema','table'])
df_model_names = pd.DataFrame(de_model_names, columns = ["DATABASE", "OLD_OBJECT","NEW_OBJECT"])
df_common_vs_pub = pd.DataFrame(de_common_vs_pub, columns = ["COMMON","PUB"])

final_df_models = pd.concat([final_df_models, df_models], ignore_index=True)
final_df_sources = pd.concat([final_df_sources, df_sources], ignore_index=True)
final_df_model_names = pd.concat([final_df_model_names, df_model_names], ignore_index=True)
final_df_common_vs_pub = pd.concat([final_df_common_vs_pub, df_common_vs_pub], ignore_index=True)

In [ ]:
#4.Save it for future use as dbt list takes time (made it persistent)
final_df_models.to_csv("final_df_models.csv",index=False)
final_df_sources.to_csv("final_df_sources.csv",index=False)
final_df_model_names.to_csv("final_df_model_names.csv",index=False)
final_df_common_vs_pub.to_csv("final_df_common_vs_pub.csv",index=False)

In [ ]:
# Read it whenever required
final_df_models = pd.read_csv('final_df_models.csv')
final_df_sources = pd.read_csv('final_df_sources.csv')
final_df_model_names = pd.read_csv('final_df_model_names.csv')
final_df_common_vs_pub = pd.read_csv('final_df_common_vs_pub.csv')

In [ ]:
all_models_dict = {}
bronze_models_dict = {}
silver_models_dict = {}
gold_models_dict = {}

for index,row in final_df_model_names.iterrows():
    old_model = row['OLD_OBJECT']
    new_model = row['NEW_OBJECT']
    database = row['DATABASE']

    all_models_dict[old_model] = new_model

    if database == 'BRONZE':
        bronze_models_dict[old_model] = new_model
    elif database == 'SILVER':
        silver_models_dict[old_model] = new_model
    else:
        gold_models_dict[old_model] = new_model

In [ ]:
common_vs_pub_dict = {}

for index,row in final_df_common_vs_pub.iterrows():
    common = row['COMMON']
    pub = row['PUB']

    common_vs_pub_dict[common] = pub

In [ ]:
print(f'total_models - {len(all_models_dict.keys())}')
print(f'bronze_models - {len(bronze_models_dict.keys())}')
print(f'silver_models - {len(silver_models_dict.keys())}')
print(f'gold_models - {len(gold_models_dict.keys())}')

In [ ]:
#1. SET PATH FOR NEW DBT PROJECT and new schema
new_dbt_path = 'C:\\Sources\\Repo\\EnterpriseCloud\\astro_datautility\\dags\\dbt\\dbt_tel'

new_schema_name_lowered = new_schema_name.lower()

Generate ALL Objects Lineage for the DEs to be migrated Ends

Model Generation in new dbt project STARTS

In [ ]:
#2.BUILD BRONZE
for node in manifest_json['nodes'].values():
    db = node['database'].upper()
    schema = node['schema'].upper()
    table = node['alias'].upper()
    if node['name'] in bronze_models_dict.keys():
        new = bronze_models_dict[node['name']]
        new_alias = new.split('__')[-1]
        sql = node['raw_code']
        new_sql = sql.replace('\r', '')
        pattern = r"alias\s*=\s*['\"]([^'\"]+)['\"]"
        new_sql = re.sub(pattern,f"alias='{new_alias}'", new_sql)
        
        for old_schema, new_schema in schema_mappings.items():
            # Match the pattern "{{ source('old_schema', '<table_name>') }}"
            pattern = rf"source('{old_schema}'"
            new_string = f"source('{new_schema}'"
            
            new_sql = new_sql.replace(pattern, new_string)
        
        for ref in node['refs']:
            ref_model_name  = ref['name']
            if ref_model_name in all_models_dict:
                new_ref_model_name = all_models_dict[ref_model_name]
            else:
                new_ref_model_name = ref_model_name
            new_ref = "{{ ref('"+new_ref_model_name+"') }}"
            ref_regex = '''\\{\\{ *ref *\\( *['"] *'''+ref_model_name+''' *['"] *\\) *\\}\\}'''
            new_sql = re.sub(ref_regex, new_ref, new_sql)

            if 'PUB__DE_OPERATIONS' in ref_model_name:
                de_operations_source = f"source('DE_OPERATIONS', '{ref_model_name.split('__')[-1]}')"
                new_source = "{{ "+de_operations_source+" }}"
                ref_regex = '''\\{\\{ *ref *\\( *['"] *'''+ref_model_name+''' *['"] *\\) *\\}\\}'''
                new_sql = re.sub(ref_regex, new_source, new_sql)
            
        new_path = f'models\\{new_schema_name_lowered}\\bronze'
        new_path_with_schema = os.path.join(new_dbt_path,new_path)
        if not os.path.exists(new_path_with_schema):
            os.makedirs(new_path_with_schema)
        with open(os.path.join(new_path_with_schema,f'{new}.SQL'), 'w+', encoding='utf-8') as f:
            f.write(new_sql)  

        

In [ ]:
#3.BUILD SILVER

for node in manifest_json['nodes'].values():
    db = node['database'].upper()
    schema = node['schema'].upper()
    table = node['alias'].upper()
    
    if node['name'] in silver_models_dict.keys():
        
        new = silver_models_dict[node['name']]  
        new_alias = new.split('__')[-1]
        sql = node['raw_code']
        new_sql = sql.replace('\r', '')
        pattern = r"alias\s*=\s*['\"]([^'\"]+)['\"]"
        new_sql = re.sub(pattern,f"alias='{new_alias}'", new_sql)
        
        for old_schema, new_schema in schema_mappings.items():
            # Match the pattern "{{ source('old_schema', '<table_name>') }}"
            pattern = rf"source('{old_schema}'"
            new_string = f"source('{new_schema}'"
            
            new_sql = new_sql.replace(pattern, new_string)
        
        for ref in node['refs']:
            ref_model_name  = ref['name']
            if ref_model_name in all_models_dict:
                new_ref_model_name = all_models_dict[ref_model_name]
            else:
                new_ref_model_name = ref_model_name
            new_ref = "{{ ref('"+new_ref_model_name+"') }}"
            ref_regex = '''\\{\\{ *ref *\\( *['"] *'''+ref_model_name+''' *['"] *\\) *\\}\\}'''
            new_sql = re.sub(ref_regex, new_ref, new_sql)

            if 'PUB__DE_OPERATIONS' in ref_model_name:
                de_operations_source = f"source('DE_OPERATIONS', '{ref_model_name.split('__')[-1]}')"
                new_source = "{{ "+de_operations_source+" }}"
                ref_regex = '''\\{\\{ *ref *\\( *['"] *'''+ref_model_name+''' *['"] *\\) *\\}\\}'''
                new_sql = re.sub(ref_regex, new_source, new_sql)
            
        
        new_path = f'models\\{new_schema_name_lowered}\\silver'
        new_path_with_schema = os.path.join(new_dbt_path,new_path)
        if not os.path.exists(new_path_with_schema):
            os.makedirs(new_path_with_schema)
        with open(os.path.join(new_path_with_schema,f'{new}.SQL'), 'w+', encoding='utf-8') as f:
            f.write(new_sql)  

In [ ]:
#4.BUILD GOLD

for node in manifest_json['nodes'].values():
    db = node['database'].upper()
    schema = node['schema'].upper()
    table = node['alias'].upper()
    if node['name'] in gold_models_dict.keys():
        new = gold_models_dict[node['name']]
        new_alias = new.split('__')[-1]
        sql = node['raw_code']
        new_sql = sql.replace('\r', '')
        pattern = r"alias\s*=\s*['\"]([^'\"]+)['\"]"
        new_sql = re.sub(pattern,f"alias='{new_alias}'", new_sql)
        
        
        for old_schema, new_schema in schema_mappings.items():
            # Match the pattern "{{ source('old_schema', '<table_name>') }}"
            pattern = rf"source('{old_schema}'"
            new_string = f"source('{new_schema}'"
            
            new_sql = new_sql.replace(pattern, new_string)
        
        for ref in node['refs']:
            ref_model_name  = ref['name']
            if ref_model_name in all_models_dict:
                new_ref_model_name = all_models_dict[ref_model_name]
            else:
                new_ref_model_name = ref_model_name
            new_ref = "{{ ref('"+new_ref_model_name+"') }}"
            ref_regex = '''\\{\\{ *ref *\\( *['"] *'''+ref_model_name+''' *['"] *\\) *\\}\\}'''
            new_sql = re.sub(ref_regex, new_ref, new_sql)

            if 'PUB__DE_OPERATIONS' in ref_model_name:
                de_operations_source = f"source('DE_OPERATIONS', '{ref_model_name.split('__')[-1]}')"
                new_source = "{{ "+de_operations_source+" }}"
                ref_regex = '''\\{\\{ *ref *\\( *['"] *'''+ref_model_name+''' *['"] *\\) *\\}\\}'''
                new_sql = re.sub(ref_regex, new_source, new_sql)
        
        new_path = f'models\\{new_schema_name_lowered}\\gold'
        new_path_with_schema = os.path.join(new_dbt_path,new_path)
        if not os.path.exists(new_path_with_schema):
            os.makedirs(new_path_with_schema)
        with open(os.path.join(new_path_with_schema,f'{new}.SQL'), 'w+', encoding='utf-8') as f:
            f.write(new_sql)  


Build new Objects in New Databases (Clone Prod): Ends

New dbt Tests Generation Starts

Refer to [this wiki](https://dev.azure.com/tokyoelectron/EnterpriseDataWarehouse/_wiki/wikis/EnterpriseDataWarehouse.wiki/851/Data-Quality-Framework) for more details

In [ ]:
class NoAliasDumper(yaml.SafeDumper):
    def ignore_aliases(self, data):
        return True

### Inputs ! 
- Repository Path
- Schema Name of New Arch
- Manifest File path 
- User Email to Query Snowflake if Needed

### Automation For Adding Default Test Cases in Bronze, Silver and Gold.

In [ ]:
#2.LOAD DBT OBJECT
from dbt.cli.main import dbtRunner,dbtRunnerResult
from dbt.contracts.graph.manifest import Manifest

res: dbtRunnerResult = dbtRunner().invoke(["parse","--project-dir",new_dbt_path])
manifest: Manifest = res.result

dbt = dbtRunner(manifest=manifest)

In [ ]:
manifest_path = os.path.join(new_dbt_path,"target\\manifest.json")
email_user = "gaurav.x.saxena@us.tel.com"
base_path = Path(new_dbt_path)

In [ ]:
with open(manifest_path, 'r') as f:
    manifest = json.load(f)

layers = ['bronze','silver','gold']

for item in layers:

    models_path = f'models\\{new_schema_name_lowered}\\{item}\\'
   
    inputModels = {node['name']: 1 for _ , node in manifest['nodes'].items() if node['resource_type'] == 'model' and node['original_file_path'].startswith(models_path)}

    inputYamlPath = base_path.joinpath(f'{models_path}_models.yml')

    inputModelTests = [
        'recently_run',
        'has_data',
    ]
    if item == 'bronze':
        inputColumnsTests = [
            { 
                'name': '_FILEDATE',
                'description': 'Timestamp of ingestion job start',
                'tests': [
                    {
                        'dbt_expectations.expect_row_values_to_have_recent_data': {
                            'datepart': 'hours',
                            'interval': 24
                        },
                    }
                ]
            },
        ]
    else:
        inputColumnsTests = []

    if os.path.exists(inputYamlPath):
        with open(inputYamlPath,'r') as y:
            yml_file = yaml.safe_load(y)
    else:
        yml_file = {'models': []}

    # first loops through the existing yaml, making sure to leave existing configs untouched
    for idx, model in enumerate(yml_file.get('models', [])):
        existing_model_tests = model.get('tests', [])
        existing_columns = model.get('columns', [])
        # loops through the model tests to be added if not already there
        for new_model_test in inputModelTests:
            if new_model_test not in existing_model_tests:
                existing_model_tests.append(new_model_test)
        # loops through columns/tests to be added 
        for new_col in inputColumnsTests:
            col_existed = False
            # loops though the existing model column's configs
            for col_idx, col in enumerate(existing_columns):
                if new_col['name'] == col['name']:
                    col_existed = True
                    existing_col_tests = col.get('tests', [])
                    # loops through the column tests to be added if not already there
                    for new_col_test in new_col['tests']:
                        if new_col_test not in existing_col_tests:
                            existing_col_tests.append(new_col_test)
                    # sorts the column tests by their key
                    col['tests'] = sorted(existing_col_tests, key=lambda d: list(d.keys())[0] if isinstance(d, dict) else d)
            if not col_existed:
                existing_columns.append(new_col)
        # sorts the model tests by their key
        model['tests'] = sorted(existing_model_tests, key=lambda d: list(d.keys())[0] if isinstance(d, dict) else d)
        
        if 'columns' in model:         
            # temporarily removes the columns to ensure they are placed at the bottom of the model's yaml
            del model['columns']
        # sorts the columns by their name ensuring underscores are placed at the top
        model['columns'] = sorted(existing_columns, key=lambda d: d.get('name', '').lower().replace('_', 'a'*9))
        # we won't need to process this model again so it can be removed from inputModels
        if model['name'] in inputModels:
            del inputModels[model['name']]
        
    # add models that weren't already part of the yaml
    for model_name in inputModels:
        input_yml = {
            'name': model_name,
            'tests': sorted(inputModelTests, key=lambda d: list(d.keys())[0] if isinstance(d, dict) else d),
        }
        if inputColumnsTests:
            input_yml['columns'] = inputColumnsTests
        yml_file['models'].append(input_yml)

    # sorts all the models by their name
    yml_file['models'] = sorted(yml_file['models'], key=lambda d: d.get('name', ''))

    with open(inputYamlPath,'w') as c:
        yaml.dump(yml_file ,c,default_flow_style=False,Dumper=NoAliasDumper,sort_keys = False)

### Automation For Bronze Test Cases.

---

This automation attempts the following:

#### Refinement Logic for Freshness Test Automation

- **Check if** `FILEDATE`, `CAPTUREDATETIME`, or `LOADDATETIME` fields are available in the model.

- **If these fields are present:**
  - Analyze the data load pattern.
  - If the data loads **consistently** (e.g., daily), **retain** the freshness check.
  - If the data loads **inconsistently** (e.g., sporadic, non-daily loads), **remove** the freshness check (or adjust it based on a specific pattern if observed).

- **If none of the fields are found:**  
  - **Remove** the freshness check for that model.  

---

> **Note:**  
> This does **not** mean we are removing the tests permanently.  
> Freshness checks will be applied iteratively where the model behavior justifies it.

---


In [ ]:
connection_details = {
    'user' : email_user,
    'account' : "telus-global.privatelink",
    'client_session_keep_alive' : False,
    'warehouse' : "DEVELOPER_WH",
    'database' : 'BRONZE',
    'role' : 'SYSADMIN_DEV',
    'schema' : 'PUBLIC',
    'authenticator' : "externalbrowser"
}

connection = snowflake.connector.connect(**connection_details)

bronze_yaml_path = base_path / 'models' / new_schema_name_lowered  / 'bronze' / '_models.yml'
model_path = base_path / 'models' / new_schema_name_lowered  / 'bronze' 
_scr_path = base_path / 'models' /  '_src'

list_of_models = [
    os.path.join(model_path, model) for model in os.listdir(model_path)  
]

ENV = 'DEV'

pattern = r'''(?i)\bFROM\b\s+\{\{\s*source\(\s*["']([^"']+)["']\s*,\s*["']([^"']+)["']\s*\)\s*\}\}'''

In [ ]:
def search_file_name(list_of_models, file):
    return next(
        item for item in list_of_models
        if re.search(
            pattern = f'{file}',
            string =item
        )
    )

def get_fqn_and_query(text):

    splits = re.split(
        pattern = r'__',
        string = text 
    )

    if ENV != 'PROD':

        query = f"""
            select 
                COLUMN_NAME 
            FROM {splits[-3]}_{ENV}.INFORMATION_SCHEMA.COLUMNS 
                WHERE TABLE_NAME ILIKE '%{splits[-1]}' AND TABLE_SCHEMA = '{splits[-2]}' ;
        """

        return f"{splits[-3]}_{ENV}.{splits[-2]}.{splits[-1]}" , query
    
    else:
        query = f"""
            select 
                COLUMN_NAME 
            FROM {splits[-3]}.INFORMATION_SCHEMA.COLUMNS 
                WHERE TABLE_NAME ILIKE '%{splits[-1]}' AND TABLE_SCHEMA = '{splits[-2]}' ;
        """

        return f"{splits[-3]}.{splits[-2]}.{splits[-1]}" , query    

def search_in_src_filename(match):
    for item in os.listdir(_scr_path):
        with open(os.path.join(_scr_path,item),'r') as f:
            yaml_dict = yaml.safe_load(f.read()).get('sources')
            for schema in yaml_dict:
                if schema.get('name').lower() == match.group(1).lower():
                    wanted_schema = schema.get('schema').upper()
                    for dx in schema.get('tables'):
                        if dx.get('name').lower() == match.group(2).lower():
                            wanted_table = dx.get('identifier').upper()
                            return wanted_schema,wanted_table  

def query_to_check(column_, schema , Name, result, database):
    query = f'''
        with LZ_data AS ( 
            select DISTINCT 
                TO_DATE({column_}) AS {column_},
                YEAR(TO_DATE({column_})) AS year
            FROM {database}.{schema}.{Name}
            WHERE {column_} >= DATEADD(MONTH, -1, CURRENT_DATE) 
        ),

        daliy_run AS (
            select 
                year,
                {column_},
                DAYNAME({column_}) AS days,
                DATEDIFF(DAY, {column_},
                    LAG({column_}) OVER (PARTITION BY year ORDER BY {column_})
                ) AS DAY_DIFF,
                CASE 
                    WHEN DAY_DIFF = -1 THEN 'X'
                    WHEN DAY_DIFF IS NULL THEN 'X'
                    ELSE '' 
                end AS daily 
            FROM LZ_data 
        ),

        final_result AS (
            select 
                year, 
                COUNT(CASE WHEN daily = '' THEN 1 END) AS blank_count,
                COUNT(CASE WHEN daily <> '' THEN 1 END) AS non_blank_count,
                COUNT(*) AS total_runs,
                CASE 
                    WHEN COUNT(*) = 1 THEN 'GO'
                    WHEN COUNT(*) < 20 THEN 'STOP'
                    WHEN COUNT(CASE WHEN daily = '' THEN 1 END) > 5 
                        AND COUNT(CASE WHEN daily = '' THEN 1 END) >= COUNT(CASE WHEN daily <> '' THEN 1 END)
                    THEN 'STOP' 
                    ELSE 'GO' 
                end AS result
            FROM daliy_run
            GROUP BY year
            ORDER BY year DESC
        )      
    '''
    if result:
        return query + 'select RESULT FROM final_result;' 
    else:
        return query + 'select * FROM daliy_run order by _CAPTUREDATETIME desc limit 20;'   

In [ ]:
with open(bronze_yaml_path,'r') as f:    
    dictonary_of_bronze = yaml.safe_load(f.read())
    dictonary_of_bronze_edit = dictonary_of_bronze.copy()
    cursor = connection.cursor()
    list_of_items = []
    for item in dictonary_of_bronze_edit.get('models'):
        try:
            path_of_model = search_file_name(list_of_models,item.get('name'))
            dictonary = dict()
            dictonary['name'] = item.get('name')
            dictonary['tests'] = ['has_data', 'recently_run']

            name, query = get_fqn_and_query(item.get('name'))
            cursor.execute(query)
            columns_in_view = [ column for item in cursor.fetchall() for column in item ]

            if '_FILEDATE' in columns_in_view:
                cursor.execute(f"Select max(_FILEDATE) from {name};")
                results_date = [ column for item in cursor.fetchall() for column in item ]

                with open(path_of_model,'r') as f:
                    data_in_model = f.read()

                match = re.search(pattern, data_in_model)

                if match:

                    schema,Name = search_in_src_filename(match)
                    check_query = query_to_check('_FILEDATE', schema , Name, True, 'LZ_PROD')
                    cursor.execute(check_query)
                    results = [
                        column
                        for item in cursor.fetchall() 
                        for column in item
                    ]

                    if results and 'STOP' not in results  :
                        proper_name = re.sub(pattern=r'_DEV|_QA|_SIT', string=name, repl= '')

                        dictonary['columns'] = [
                            {
                                'name': '_FILEDATE', 
                                'description': f'Check the _FILEDATE column in the {proper_name} table to ensure it falls within the last 24-hour interval.', 
                                'tests': [
                                    {'dbt_expectations.expect_row_values_to_have_recent_data': {
                                        'datepart': 'hours', 'interval': 24
                                        }
                                    }
                                ]
                            }
                        ]
                        markdown = f'''
Processed Model => {name}
File Date for {name} is {results_date[-1].isoformat()}
Added _FILEDATE Check for Model {name}
Model Source is => {schema} {Name}
'''
                        print(markdown)  

                    else:
                        print('\n')
                        check_query = query_to_check('_FILEDATE', schema , Name, False, 'LZ_PROD')

            elif '_CAPTUREDATETIME' in columns_in_view:
                with open(path_of_model,'r') as f:
                    data_in_model = f.read()

                match = re.search(pattern, data_in_model)

                if match:
                    
                    check_query = query_to_check('_CAPTUREDATETIME', match.group(1) , match.group(2), True, 'LZ_HVR')
                    cursor.execute(check_query)
                    results = [
                        column
                        for item in cursor.fetchall() 
                        for column in item
                    ]

                    if results and 'STOP' not in results  :
                        dictonary['columns'] = [
                            {
                                'name': '_CAPTUREDATETIME', 
                                'description': f'Check the _CAPTUREDATETIME column in the {name} table to ensure it falls within the last 24-hour interval.', 
                                'tests': [
                                    {'dbt_expectations.expect_row_values_to_have_recent_data': {
                                        'datepart': 'hours', 'interval': 24
                                        }
                                    }
                                ]
                            }
                        ]
                        markdown = f'''
Processed Model => {name}
Added _CAPTUREDATETIME to {name}
Model Source is => { match.group(1)}  {match.group(2)}
'''
                        print(markdown)   

                    else:
                        print(f'''Model has Gaps in their Data Flow {name}''')
                        check_query = query_to_check('_CAPTUREDATETIME', match.group(1) , match.group(2), False, 'LZ_HVR')
                        df = pd.read_sql(check_query, connection)
                        print(df.to_markdown(index=False))
                        print('\n')
                    
                else:
                    print('Cannot Regex the Below SAPO Model Please Check')
                    print(path_of_model)

            elif 'SNAPSHOT_DATE' in columns_in_view:
                    print(f"Added SNAPSHOT_DATE Test to {name}")
                    dictonary['columns'] = [
                        {
                            'name': 'SNAPSHOT_DATE', 
                            'description': f'Check the SNAPSHOT_DATE column in the {name} table to ensure it falls within the last 24-hour interval.', 
                            'tests': [
                                {'dbt_expectations.expect_row_values_to_have_recent_data': {
                                    'datepart': 'hours', 'interval': 24
                                    }
                                }
                            ]
                        }
                    ]                
            else:
                print(f'Test Removed for {name}')  
                              
            list_of_items.append(dictonary)

        except Exception as e:
            print(name, e)

        

In [ ]:
dictonary_of_bronze['models'] = list_of_items

with open(bronze_yaml_path,'w') as f:
    yaml.safe_dump(dictonary_of_bronze, f, sort_keys=False)

New dbt Tests Generation Ends

Build new Objects in New Databases (Clone Prod) Starts

In [ ]:
#CLONING: basically Build new Objects in New Databases. (Pls replace silver bronze gold to approariate environment _dev/_qa/_sit)
final_clone = []
# exclusion = ['BRONZE__TEA_SERVICE__TIRS_ACCIDENTREPORTSFORUS_NOLOCK','SILVER__TEA_SERVICE__ACCIDENTREPORTSFORUS_NOLOCK',
#              'BRONZE__TEA_SERVICE__SAPAPI_ZCDV_T_CMTU024_ACDOCA','SILVER__TEA_SERVICE__UNIVERSAL_JOURNAL_ENTRY_LINE'
#              ,'SILVER__TEA_SERVICE__UNIVERSAL_PL_ACCOUNT_TYPE','BRONZE__TEA_SERVICE__CRM_FILE_OPPORTUNITY']
exclusion = []
for old,new in all_models_dict.items():
    if new in exclusion:
        continue 
    new_db,new_schema,new_table = new.split('__')
    old_db,old_schema,old_table = old.split('__')
    clone_sql = f"""create or replace table {new_db}.{new_schema}.{new_table} clone {old_db}.{old_schema}.{old_table};"""
    final_clone.append(clone_sql)

print(final_clone)

In [ ]:
#RUN CLONES IN SNOWFLAKE
conn = snowflake.connector.connect(user= email_user,
                                    account="telus-global.privatelink", 
                                    client_session_keep_alive = False,
                                    warehouse= "PLATFORM_WH", 
                                    role= "PLATFORM_TEAM_DEV",
                                    threads= 1,    
                                    authenticator="externalbrowser",
                                    autocommit=True)  

cursor = conn.cursor()

for sql in final_clone:
    try:
        cursor.execute(sql)
    except Exception as e:
        print(f"FAILED SQL : {sql}")

conn.close()

Build new Objects in New Databases (Clone Prod) Ends

Create OLD V/s NEW MAPPING Document Starts

In [ ]:
#1.CREATE OLD VS NEW MAPPING SHEET
mapping_dict = {}
for old,new in all_models_dict.items():
    for common, pub in common_vs_pub_dict.items(): #result_dict : is pub-common mapping generated in creating GOLD model step
        if old == common:
            mapping_dict[pub] = new
            break
    mapping_dict[old] = new

mapping_df = []
for old,new in mapping_dict.items():
    new_db,new_schema,new_table = new.split('__')
    old_db,old_schema,old_table = old.split('__')
    old_obj =f'{old_db}.{old_schema}.{old_table}'
    new_obj =f'{new_db}.{new_schema}.{new_table}'
    mapping_df.append((old_db,old_schema,old_table,old_obj,new_db,new_schema,new_table,new_obj))
    

df_map = pd.DataFrame(mapping_df, columns = ["OLD DATABASE","OLD SCHEMA","OLD TABLE","OLD OBJ","NEW DATABASE","NEW SCHEMA","NEW TABLE","NEW OBJ"])
df_map.to_csv(f"{new_schema_name_lowered}_Mapping_Document.csv")

Create OLD V/s NEW MAPPING Document Ends

Parrallel Testing Starts

In [ ]:
#parrallel testing 
df = pd.read_csv("C:/Sources/Repo/EnterpriseCloud/Notebooks/teh_ehs_Mapping_Document.csv")
df['OLD COUNT'] = None
df['NEW COUNT'] = None
new_db_env = "DEV"

conn = snowflake.connector.connect(user= email_user,
                                    account="telus-global.privatelink", 
                                    client_session_keep_alive = False,
                                    warehouse= "DEVELOPER_WH", 
                                    role= "SYSADMIN_DEV",
                                    threads= 1,    
                                    authenticator="externalbrowser",
                                    autocommit=True)  

cursor = conn.cursor()
for idx,row in df.iterrows():
    try:
        old_count_sql = f"""select count(*) from {row["OLD DATABASE"]}.{row["OLD SCHEMA"]}.{row["OLD TABLE"]}"""            
        cursor.execute(old_count_sql)
        old_cn = cursor.fetchone()[0]
        df.at[idx, 'OLD COUNT'] = old_cn
    except Exception as e:
        print(e) 

cursor = conn.cursor()
for idx,row in df.iterrows():
    try:
        new_count_sql = f"""select count(*) from {row["NEW DATABASE"]}_{new_db_env}.{row["NEW SCHEMA"]}.{row["NEW TABLE"]}"""
        cursor.execute(new_count_sql)
        new_cn = cursor.fetchone()[0]
        df.at[idx, 'NEW COUNT'] = new_cn
    except Exception as e:
        print(e)

conn.close()

df_counts = duckdb.query("""select * , ROUND(
        (ABS("OLD COUNT"-"NEW COUNT") / 
        GREATEST("OLD COUNT","NEW COUNT"))*100
        ,2) AS COUNT_DIFF from df""").df()  

In [ ]:
df_counts.to_excel(f"{new_schema_name_lowered}_counts_{datetime.now():_%m_%d}.xlsx", index=False, engine='openpyxl')   
print(f"Data exported to {new_schema_name_lowered}_counts_{datetime.now():%m_%d}.xlsx")  

Parrallel Testing Ends

Snow file Generation Starts

In [ ]:
df_tables = pd.read_csv(f"{new_schema_name_lowered}_Mapping_Document.csv")
table_ddls__path = r"C:\Sources\Repo\EnterpriseCloud\Snow"
role = "PLATFORM_TEAM"
ticket_number = "37951"

In [ ]:
conn = snowflake.connector.connect(user= email_user,
                                    account="telus-global.privatelink", 
                                    client_session_keep_alive = False,
                                    warehouse= "DEVELOPER_WH", 
                                    role= "SYSADMIN_DEV",
                                    threads= 1,    
                                    authenticator="externalbrowser",
                                    autocommit=True) 


cursor = conn.cursor()

file_handles = {}

for index,row in df_tables.iterrows():

    database = f'{row["NEW DATABASE"]}_DEV'
    schema = row["NEW SCHEMA"]
    table = row["NEW TABLE"]

    oldDatabase = f'{row["OLD DATABASE"]}'
    oldSchema = row["OLD SCHEMA"]
    oldTable = row["OLD TABLE"]

    try:
        if database not in file_handles:
            ticket_description = f"{schema}_{database.replace('_DEV','')}"
            file_path = os.path.join(table_ddls__path, f"k.{ticket_number}_{ticket_description}.sql")

            file_handles[database] = open(file_path, "w", encoding="utf-8")
            file_handles[database].write(f"USE ROLE <<{role}>>;\n\n")
            file_handles[database].write(f"USE DATABASE <<{database.replace('_DEV', '')}>>;\n\n")
            file_handles[database].write(f"USE SCHEMA {schema};\n\n")

        f = file_handles[database]    

        ddl_query = f"SELECT GET_DDL('TABLE','{database}.{schema}.{table}')"

        cursor.execute(ddl_query)
        result = cursor.fetchone()

        if result:
            f.write(result[0])
            f.write("\n\n")
        else:
            print(f"No DDL found for {database}.{schema}.{table} ...\n")
    
    except Exception as e:
        print(f"Error fetching DDL for {database}.{schema}.{table}: {e} \n")

for db, f in file_handles.items():
    f.close()

cursor.close()   
conn.close() 

print(f"All DDLs saved to {table_ddls__path}")

Snow file Generation Ends